# Monte Carlo π with nb2slurm

A complete, runnable nb2slurm example. The workflow estimates π by Monte Carlo:
throw random darts at the unit square and count how many land inside the quarter
circle. We run it for several sample sizes and random seeds — each
`(n_samples, seed)` pair is one SLURM job.

This control notebook builds the scripts, proves the workflow runs locally on one
job, and shows how to scale it out to SLURM. The workflow notebooks live in
`notebooks/`, the helper in `scripts/montecarlo.py`, and the jobs in `jobs.json`.

## 1. Describe the workflow

`varying=["n_samples", "seed"]` matches the two levels in `jobs.json`. We keep
the default `python3` kernel so this runs locally with no setup.

In [ ]:
import nb2slurm

wf = nb2slurm.Workflow(
    name="monte_carlo_pi",
    notebooks=[
        "notebooks/0_settings.ipynb",
        "notebooks/1_simulate.ipynb",
        "notebooks/2_plot.ipynb",
    ],
    kernel="python3",
    varying=["n_samples", "seed"],
    jobs_json="jobs.json",
    resources=dict(nodes=1, cpus=1, time="00:05:00"),
)

## 2. The jobs

`jobs.json` already holds the grid; read it back to see the jobs it expands to.

In [ ]:
for job in wf.jobs_from_json():
    print(job)
# -> ('1000', '1') ('1000', '2') ('1000', '3') ('1000000', '1') ...

## 3. Generate the scripts

Renders `scripts/run_workflow.py`, `job.slurm`, the submitters and `jobs.txt`.

In [ ]:
for kind, path in wf.build().items():
    print(f"{kind:13s} -> {path}")

## 4. Run ONE job locally (the duality of use)

Before any cluster, prove the chain works on a single job by calling the
generated runner directly. This creates `output/1000/1/` with the executed
notebooks, `settings.json`, `result.json` and `scatter.png`.

In [ ]:
import subprocess, sys
proc = subprocess.run(
    [sys.executable, "scripts/run_workflow.py", "1000", "1"],
    capture_output=True, text=True,
)
print(proc.stdout)
print(proc.stderr[-2000:] if proc.returncode else "OK")

## 5. Inspect the result

In [ ]:
import json
from pathlib import Path
result = json.loads(Path("output/1000/1/result.json").read_text())
print(result)

from IPython.display import Image
Image(filename="output/1000/1/scatter.png")

## 6. See what would be submitted to SLURM

`submit(dry_run=True)` prints the exact `sbatch` commands (one per job) without
running them — no cluster needed.

In [ ]:
wf.submit(dry_run=True)

## 7. Scale out on a real cluster

On an actual HPC you'd connect over SSH and let nb2slurm do the rest. (Left
commented since there's no cluster here.)

In [ ]:
# cfg = nb2slurm.SSHConfig(host="spider.surf.nl", user="me",
#                          remote_dir="/home/me/monte_carlo_pi",
#                          key_filename="~/.ssh/id_ed25519")
# wf.push(ssh=cfg)        # upload source (never output/)
# wf.check(ssh=cfg)       # preflight
# wf.submit(ssh=cfg)      # one job per (n_samples, seed)
# wf.status(ssh=cfg)      # monitor
# wf.pull(ssh=cfg)        # bring results back (never your notebooks)

## What made this nb2slurm-compatible?

* `0_settings.ipynb` has a `parameters` cell (`n_samples`, `seed`, `outdir`) and
  writes `settings.json`.
* `1_simulate` / `2_plot` have a `settings_path` parameters cell and load it.
* All outputs are written under `settings["outdir"]`, so parallel jobs don't
  collide.
* The simulate step is idempotent (skips if `result.json` exists).
* The helper is imported from `scripts/` with a `sys.path` fallback.
* The plot uses a headless backend + `display()`.

See `docs/setup_notebooks.ipynb` for the general checklist.